# Structured Chunk Ingestion into Qdrant

This notebook ingests structured chunk records produced by the chunk-building notebook into a Qdrant vector database.

The notebook is responsible for:
1. loading structured chunk JSON and optional audit outputs
2. validating the updated chunk schema and metadata contract
3. inspecting chunk quality, distribution, and ingestion readiness
4. verifying AWS Bedrock and Qdrant connectivity
5. preparing chunk text for embedding
6. generating embeddings for validated chunks
7. building Qdrant point payloads with rich provenance metadata
8. upserting vectors into Qdrant in controlled batches
9. saving ingestion audit outputs and reporting final ingestion results

This notebook is designed for:
- chunk records generated from full-document XBRL parser output
- multiple chunk types such as filing metadata, raw fact groups, unified metric groups, and narrative disclosures
- commercial / NBFC / banking / insurance filings
- preservation of taxonomy, context, section, and source-file provenance in vector payloads
- safe reruns through deterministic point IDs and collection reuse checks

This notebook does **not** perform retrieval testing, ranking evaluation, or search quality assessment.
That should be done in a separate retrieval and evaluation notebook.

## 1. Import Required Libraries

In [1]:
import json
import re
import time
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from collections import Counter

import boto3
from botocore.exceptions import ClientError

from tqdm import tqdm

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

## 1. Define Ingestion Configuration

In [ ]:
CONFIG = {
    "input_file_path": "../data/processed/chunks/financial_structured_chunks_v2.json",
    "input_audit_file_path": "../data/processed/audits/financial_structured_chunks_audit_v2.json",
    "aws_region": "us-east-1",
    "embed_model_id": "amazon.titan-embed-text-v2:0",
    "embed_dimension": 1024,
    "embed_normalize": True,
    "embed_max_retries": 5,
    "embed_text_max_chars": 12000,
    "qdrant_host": "localhost",
    "qdrant_port": 6333,
    "collection_name": "financial_chunks_xbrl_context_aware_v1",
    "batch_size": 50,
    "allowed_chunk_types": None,
    "allowed_taxonomies": None,
    "allowed_company_names": None,
    "allowed_source_files": None,
    "max_chunks": None,
    "ingestion_audit_output_path": "../data/processed/audits/qdrant_ingestion_audit_v1.json",
    "failed_chunks_output_path": "../data/processed/audits/qdrant_failed_chunks_v1.json",
    "schema_issues_output_path": "../data/processed/audits/qdrant_schema_issues_v1.json",
}

print("Pipeline configuration loaded successfully.\n")
for key, value in CONFIG.items():
    print(f"{key}: {value}")

Pipeline configuration loaded successfully.

input_file_path: ../data/processed/chunks/financial_structured_chunks_v2.json
input_audit_file_path: ../data/processed/audits/financial_structured_chunks_audit_v2.json
aws_region: us-east-1
embed_model_id: amazon.titan-embed-text-v2:0
embed_dimension: 1024
embed_normalize: True
embed_max_retries: 5
embed_text_max_chars: 12000
qdrant_host: localhost
qdrant_port: 6333
collection_name: financial_chunks_xbrl_context_aware_v1
batch_size: 50
allowed_chunk_types: None
allowed_taxonomies: None
allowed_company_names: None
allowed_source_files: None
max_chunks: None
ingestion_audit_output_path: ../data/processed/audits/qdrant_ingestion_audit_v1.json
failed_chunks_output_path: ../data/processed/audits/qdrant_failed_chunks_v1.json
schema_issues_output_path: ../data/processed/audits/qdrant_schema_issues_v1.json


## 2. Define Base File and Utility Helpers

In [ ]:
def load_json(json_path: Path) -> Any:
    path = Path(json_path)
    if not path.exists():
        raise FileNotFoundError(f"JSON file not found: {path.resolve()}")
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def save_json(data: Any, output_path: Path) -> Path:
    path = Path(output_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=2, ensure_ascii=False)
    return path


def count_words(text: Any) -> int:
    return len(re.findall(r"\b\w+\b", str(text)))


def short_hash(text: Any, length: int = 12) -> str:
    return hashlib.md5(str(text).encode("utf-8")).hexdigest()[:length]


def normalize_filter_values(values: Optional[List[str]]) -> Optional[set]:
    if not values:
        return None
    return {str(value).strip() for value in values if str(value).strip()}


def sanitize_payload_value(value: Any) -> Any:
    if value is None:
        return None
    if isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, list):
        sanitized_list = []
        for item in value:
            if isinstance(item, (str, int, float, bool)) or item is None:
                sanitized_list.append(item)
            else:
                sanitized_list.append(str(item))
        return sanitized_list
    if isinstance(value, dict):
        sanitized_dict = {}
        for key, item in value.items():
            if isinstance(item, (str, int, float, bool)) or item is None:
                sanitized_dict[str(key)] = item
            elif isinstance(item, list):
                sanitized_dict[str(key)] = sanitize_payload_value(item)
            else:
                sanitized_dict[str(key)] = str(item)
        return sanitized_dict
    return str(value)


print("Utility helpers ready.")

Utility helpers ready.


## 3. Load Structured Chunk JSON

In [ ]:
input_file_path = Path(CONFIG["input_file_path"])

if not input_file_path.exists():
    raise FileNotFoundError(
        f"Input file not found: {input_file_path.resolve()}\n"
        "Please run the chunk-building notebook first and verify the output path."
    )

structured_chunks = load_json(input_file_path)

if not isinstance(structured_chunks, list):
    raise ValueError(
        f"Expected input chunk file to contain a list of chunk records, "
        f"but got {type(structured_chunks).__name__}."
    )

print("Structured chunk file loaded successfully.")
print("Input file:", input_file_path.resolve())
print("Total chunk records loaded:", len(structured_chunks))

if structured_chunks:
    print("\nSample chunk record preview:")
    print(json.dumps(structured_chunks[0], indent=2, ensure_ascii=False)[:1500])
else:
    print("Warning: No chunk records found in the input file.")

Structured chunk file loaded successfully.
Input file: C:\Users\Home\llmops-nse-rag\data\processed\chunks\financial_structured_chunks_v2.json
Total chunk records loaded: 1383

Sample chunk record preview:
{
  "chunk_id": "gi-117338-1350247-17012025074725-f7b38fa1cd4c",
  "text": "Company Name: ICICI Lombard General Insurance Company Limited\nTaxonomy: INSURANCE\nReport Type: Standalone\nReporting Period: Full Filing\nContext Type: Document\nSection: Filing Metadata\n- Company Name: ICICI Lombard General Insurance Company Limited\n- Taxonomy: INSURANCE\n- Report Type: Standalone\n- Presentation Currency: INR\n- Rounding: Unknown Rounding\n- Source File: GI_117338_1350247_17012025074725.xml",
  "metadata": {
    "company_name": "ICICI Lombard General Insurance Company Limited",
    "currency": "INR",
    "rounding": "Unknown Rounding",
    "report_type": "Standalone",
    "source_file": "GI_117338_1350247_17012025074725.xml",
    "taxonomy": "INSURANCE",
    "section_key": "filing_metada

## 4. Load Optional Chunk Audit Output

In [ ]:
input_audit_file_path = Path(CONFIG["input_audit_file_path"])
chunk_audit_payload = None
chunk_audit_summary = {}

if input_audit_file_path.exists():
    chunk_audit_payload = load_json(input_audit_file_path)

    chunk_audit_summary = {
        "files_discovered": chunk_audit_payload.get("files_discovered"),
        "files_processed_successfully": chunk_audit_payload.get("files_processed_successfully"),
        "files_failed": chunk_audit_payload.get("files_failed"),
        "total_chunks": chunk_audit_payload.get("total_chunks"),
        "validation": chunk_audit_payload.get("validation", {}),
    }

    print("Upstream chunk audit file loaded successfully.")
    print("Audit file:", input_audit_file_path.resolve())
    print(json.dumps(chunk_audit_summary, indent=2, ensure_ascii=False))
else:
    print("No upstream chunk audit file found. Proceeding with chunk JSON only.")
    print("Expected audit path:", input_audit_file_path.resolve())

No upstream chunk audit file found. Proceeding with chunk JSON only.
Expected audit path: C:\Users\Home\llmops-nse-rag\data\processed\audits\financial_structured_chunks_audit_v2.json


## 5. Inspect Input Chunk Contract

In [6]:
sample_metadata_keys = []
sample_top_level_keys = []
sample_chunk_types = Counter()

for chunk in structured_chunks[:5]:
    if isinstance(chunk, dict):
        sample_top_level_keys.append(sorted(chunk.keys()))
        metadata = chunk.get("metadata", {})
        if isinstance(metadata, dict):
            sample_metadata_keys.append(sorted(metadata.keys()))
            sample_chunk_types[metadata.get("chunk_type", "Unknown")] += 1

print("Input contract inspection")
print("-" * 80)
print("Top-level key samples:")
for keys in sample_top_level_keys[:3]:
    print(keys)

print("\nMetadata key samples:")
for keys in sample_metadata_keys[:3]:
    print(keys)

print("\nSample chunk type counts from first records:")
for chunk_type, count in sample_chunk_types.items():
    print(f"- {chunk_type}: {count}")

Input contract inspection
--------------------------------------------------------------------------------
Top-level key samples:
['chunk_id', 'metadata', 'text']
['chunk_id', 'metadata', 'text']
['chunk_id', 'metadata', 'text']

Metadata key samples:
['chunk_index', 'chunk_type', 'company_name', 'context_ids', 'context_signature', 'context_type', 'currency', 'dimension_members', 'end_date', 'instant_date', 'markdown_style', 'report_type', 'reporting_period', 'rounding', 'section_key', 'section_name', 'source_file', 'source_tags', 'start_date', 'taxonomy', 'unified_fields', 'word_count']
['chunk_index', 'chunk_type', 'company_name', 'context_ids', 'context_signature', 'context_type', 'currency', 'dimension_members', 'end_date', 'instant_date', 'markdown_style', 'report_type', 'reporting_period', 'rounding', 'section_key', 'section_name', 'source_file', 'source_tags', 'start_date', 'taxonomy', 'unified_fields', 'word_count']
['chunk_index', 'chunk_type', 'company_name', 'context_ids', '

## 6. Define Expected Chunk Schema for Ingestion

In [7]:
required_top_level_fields = {"chunk_id", "text", "metadata"}

required_metadata_fields = {
    "company_name",
    "currency",
    "rounding",
    "report_type",
    "source_file",
    "taxonomy",
    "section_key",
    "section_name",
    "chunk_type",
    "reporting_period",
    "context_type",
    "context_ids",
    "source_tags",
    "unified_fields",
    "context_signature",
    "chunk_index",
    "word_count",
}

required_list_metadata_fields = {
    "context_ids",
    "source_tags",
    "unified_fields",
}

optional_list_metadata_fields = {
    "dimension_members",
}

print("Expected schema defined successfully.")
print("Required top-level fields:", sorted(required_top_level_fields))
print("Required metadata fields:", sorted(required_metadata_fields))

Expected schema defined successfully.
Required top-level fields: ['chunk_id', 'metadata', 'text']
Required metadata fields: ['chunk_index', 'chunk_type', 'company_name', 'context_ids', 'context_signature', 'context_type', 'currency', 'report_type', 'reporting_period', 'rounding', 'section_key', 'section_name', 'source_file', 'source_tags', 'taxonomy', 'unified_fields', 'word_count']


## 7. Validate Input Schema and Required Fields

In [8]:
schema_issues: List[Dict[str, Any]] = []
valid_chunks: List[Dict[str, Any]] = []

for index, chunk in enumerate(structured_chunks):
    chunk_issues = []

    if not isinstance(chunk, dict):
        schema_issues.append({
            "record_index": index,
            "issue": f"Chunk record is not a dictionary: {type(chunk).__name__}",
        })
        continue

    missing_top_level = sorted(required_top_level_fields - set(chunk.keys()))
    if missing_top_level:
        chunk_issues.append(f"Missing top-level fields: {missing_top_level}")

    metadata = chunk.get("metadata", {})
    if not isinstance(metadata, dict):
        chunk_issues.append("Metadata is missing or is not a dictionary")
    else:
        missing_metadata = sorted(required_metadata_fields - set(metadata.keys()))
        if missing_metadata:
            chunk_issues.append(f"Missing metadata fields: {missing_metadata}")

        for field_name in required_list_metadata_fields:
            if field_name in metadata and not isinstance(metadata.get(field_name), list):
                chunk_issues.append(f"Metadata field '{field_name}' must be a list")

        for field_name in optional_list_metadata_fields:
            if field_name in metadata and metadata.get(field_name) is not None and not isinstance(metadata.get(field_name), list):
                chunk_issues.append(f"Optional metadata field '{field_name}' must be a list when present")

        if "chunk_index" in metadata and not isinstance(metadata.get("chunk_index"), int):
            chunk_issues.append("Metadata field 'chunk_index' must be an integer")

        if "word_count" in metadata and not isinstance(metadata.get("word_count"), int):
            chunk_issues.append("Metadata field 'word_count' must be an integer")

    text = chunk.get("text", "")
    if not isinstance(text, str) or not text.strip():
        chunk_issues.append("Text is missing or empty")

    chunk_id = chunk.get("chunk_id", "")
    if not isinstance(chunk_id, str) or not chunk_id.strip():
        chunk_issues.append("chunk_id is missing or empty")

    if chunk_issues:
        schema_issues.append({
            "record_index": index,
            "chunk_id": chunk.get("chunk_id"),
            "source_file": metadata.get("source_file") if isinstance(metadata, dict) else None,
            "issues": chunk_issues,
        })
    else:
        valid_chunks.append(chunk)

print("Schema validation completed.")
print("Total records checked:", len(structured_chunks))
print("Valid records:", len(valid_chunks))
print("Invalid records:", len(schema_issues))

Schema validation completed.
Total records checked: 1383
Valid records: 1383
Invalid records: 0


## 8. Review Schema Issues and Duplicate Chunk IDs

In [9]:
valid_chunk_ids = [chunk["chunk_id"] for chunk in valid_chunks]
duplicate_chunk_ids = [chunk_id for chunk_id, count in Counter(valid_chunk_ids).items() if count > 1]

print("Schema and duplicate review")
print("-" * 80)
print("Schema issues found:", len(schema_issues))
print("Duplicate chunk IDs among valid chunks:", len(duplicate_chunk_ids))

if schema_issues:
    print("\nSample schema issues:")
    for issue in schema_issues[:5]:
        print(json.dumps(issue, indent=2, ensure_ascii=False))

if duplicate_chunk_ids:
    print("\nSample duplicate chunk IDs:")
    for chunk_id in duplicate_chunk_ids[:10]:
        print("-", chunk_id)
else:
    print("\nNo duplicate chunk IDs found among valid chunks.")

Schema and duplicate review
--------------------------------------------------------------------------------
Schema issues found: 0
Duplicate chunk IDs among valid chunks: 0

No duplicate chunk IDs found among valid chunks.


## 9. Apply Ingestion Filters and Build Final Candidate Set

In [10]:
allowed_chunk_types = normalize_filter_values(CONFIG["allowed_chunk_types"])
allowed_taxonomies = normalize_filter_values(CONFIG["allowed_taxonomies"])
allowed_company_names = normalize_filter_values(CONFIG["allowed_company_names"])
allowed_source_files = normalize_filter_values(CONFIG["allowed_source_files"])
max_chunks = CONFIG["max_chunks"]

filter_summary = {
    "valid_chunks_input": len(valid_chunks),
}

candidate_chunks = valid_chunks

if allowed_chunk_types is not None:
    candidate_chunks = [
        chunk for chunk in candidate_chunks
        if str(chunk["metadata"].get("chunk_type", "")).strip() in allowed_chunk_types
    ]
filter_summary["after_chunk_type_filter"] = len(candidate_chunks)

if allowed_taxonomies is not None:
    candidate_chunks = [
        chunk for chunk in candidate_chunks
        if str(chunk["metadata"].get("taxonomy", "")).strip() in allowed_taxonomies
    ]
filter_summary["after_taxonomy_filter"] = len(candidate_chunks)

if allowed_company_names is not None:
    candidate_chunks = [
        chunk for chunk in candidate_chunks
        if str(chunk["metadata"].get("company_name", "")).strip() in allowed_company_names
    ]
filter_summary["after_company_filter"] = len(candidate_chunks)

if allowed_source_files is not None:
    candidate_chunks = [
        chunk for chunk in candidate_chunks
        if str(chunk["metadata"].get("source_file", "")).strip() in allowed_source_files
    ]
filter_summary["after_source_file_filter"] = len(candidate_chunks)

deduplicated_chunks = []
seen_chunk_ids = set()

for chunk in candidate_chunks:
    if chunk["chunk_id"] in seen_chunk_ids:
        continue
    seen_chunk_ids.add(chunk["chunk_id"])
    deduplicated_chunks.append(chunk)

filter_summary["after_deduplication"] = len(deduplicated_chunks)

if isinstance(max_chunks, int) and max_chunks > 0:
    ingestion_chunks = deduplicated_chunks[:max_chunks]
else:
    ingestion_chunks = deduplicated_chunks

filter_summary["final_ingestion_chunks"] = len(ingestion_chunks)

print("Filter summary")
print("-" * 80)
print(json.dumps(filter_summary, indent=2, ensure_ascii=False))

Filter summary
--------------------------------------------------------------------------------
{
  "valid_chunks_input": 1383,
  "after_chunk_type_filter": 1383,
  "after_taxonomy_filter": 1383,
  "after_company_filter": 1383,
  "after_source_file_filter": 1383,
  "after_deduplication": 1383,
  "final_ingestion_chunks": 1383
}


## 10. Inspect Filtered Chunk Distribution and Metadata Quality

In [11]:
company_counter = Counter()
source_file_counter = Counter()
taxonomy_counter = Counter()
chunk_type_counter = Counter()
section_counter = Counter()
reporting_period_counter = Counter()

for chunk in ingestion_chunks:
    metadata = chunk["metadata"]
    company_counter[metadata.get("company_name", "Unknown Company")] += 1
    source_file_counter[metadata.get("source_file", "Unknown Source File")] += 1
    taxonomy_counter[metadata.get("taxonomy", "UNKNOWN")] += 1
    chunk_type_counter[metadata.get("chunk_type", "Unknown Chunk Type")] += 1
    section_counter[metadata.get("section_name", "Unknown Section")] += 1
    reporting_period_counter[metadata.get("reporting_period", "Unknown Period")] += 1

ingestion_distribution = {
    "by_company": dict(company_counter),
    "by_source_file": dict(source_file_counter),
    "by_taxonomy": dict(taxonomy_counter),
    "by_chunk_type": dict(chunk_type_counter),
    "by_section": dict(section_counter),
    "by_reporting_period": dict(reporting_period_counter),
}

print("Filtered ingestion distribution")
print("-" * 80)
print("Total ingestion chunks:", len(ingestion_chunks))

print("\nChunks by chunk type:")
for key, value in chunk_type_counter.most_common():
    print(f"- {key}: {value}")

print("\nChunks by taxonomy:")
for key, value in taxonomy_counter.most_common():
    print(f"- {key}: {value}")

print("\nChunks by company:")
for key, value in company_counter.most_common(20):
    print(f"- {key}: {value}")

Filtered ingestion distribution
--------------------------------------------------------------------------------
Total ingestion chunks: 1383

Chunks by chunk type:
- raw_fact_group: 723
- unified_metric_group: 557
- narrative_disclosure: 94
- filing_metadata: 9

Chunks by taxonomy:
- INSURANCE: 668
- BANK: 381
- COMMERCIAL_NBFC: 334

Chunks by company:
- RELIANCE INDUSTRIES LIMITED: 452
- HDFC Life Insurance Company Limited: 395
- ICICI Lombard General Insurance Company Limited: 273
- HDFC Bank Limited: 124
- Bajaj Finserv Limited: 107
- Max Healthcare Institute Limited: 32


## 11. Define Text Preparation Helpers for Embedding

In [12]:
def normalize_embedding_text(text: str) -> str:
    normalized = str(text).replace("\x00", " ").replace("\ufeff", " ")
    normalized = re.sub(r"\r\n?", "\n", normalized)
    normalized = re.sub(r"[ \t]+", " ", normalized)
    normalized = re.sub(r"\n{3,}", "\n\n", normalized)
    return normalized.strip()


def prepare_text_for_embedding(text: str, max_chars: int) -> Tuple[str, Dict[str, Any]]:
    original_text = str(text)
    normalized_text = normalize_embedding_text(original_text)

    original_char_count = len(original_text)
    normalized_char_count = len(normalized_text)

    prepared_text = normalized_text
    was_truncated = False

    if len(prepared_text) > max_chars:
        truncated = prepared_text[:max_chars]
        if " " in truncated:
            truncated = truncated.rsplit(" ", 1)[0]
        prepared_text = truncated.strip()
        was_truncated = True

    preparation_metadata = {
        "original_char_count": original_char_count,
        "normalized_char_count": normalized_char_count,
        "prepared_char_count": len(prepared_text),
        "original_word_count": count_words(original_text),
        "prepared_word_count": count_words(prepared_text),
        "was_truncated": was_truncated,
    }

    return prepared_text, preparation_metadata


print("Text preparation helpers ready.")

Text preparation helpers ready.


## 12. Verify AWS Credentials and Initialize Bedrock Client

In [13]:
print("Verifying AWS access...\n")

try:
    sts_client = boto3.client("sts", region_name=CONFIG["aws_region"])
    identity = sts_client.get_caller_identity()

    bedrock_client = boto3.client(
        "bedrock-runtime",
        region_name=CONFIG["aws_region"]
    )

    print("AWS verification successful.")
    print("Bedrock client initialized successfully.")

except Exception as e:
    raise RuntimeError(
        "AWS verification failed. "
        "Please check your AWS credentials, region, and Bedrock access.\n"
        f"Details: {e}"
    )

Verifying AWS access...

AWS verification successful.
Bedrock client initialized successfully.


## 13. Verify Qdrant Connectivity

In [14]:
print("Verifying Qdrant connectivity...\n")

try:
    qdrant_client = QdrantClient(
        host=CONFIG["qdrant_host"],
        port=CONFIG["qdrant_port"]
    )

    collections_response = qdrant_client.get_collections()
    existing_collections = [collection.name for collection in collections_response.collections]

    print("Qdrant connection successful.")
    print(f"Qdrant host: {CONFIG['qdrant_host']}:{CONFIG['qdrant_port']}")
    print(f"Existing collections found: {len(existing_collections)}")

    if existing_collections:
        print("Collections:")
        for name in existing_collections:
            print("-", name)

except Exception as e:
    raise ConnectionError(
        "Qdrant verification failed. "
        "Please ensure the Qdrant server is running and reachable.\n"
        f"Details: {e}"
    )

Verifying Qdrant connectivity...

Qdrant connection successful.
Qdrant host: localhost:6333
Existing collections found: 1
Collections:
- financial_reports_chunks_v2


## 14. Define Collection Initialization and Compatibility Helpers

In [15]:
def collection_exists(client: QdrantClient, collection_name: str) -> bool:
    try:
        return client.collection_exists(collection_name)
    except AttributeError:
        collections_response = client.get_collections()
        existing_names = [collection.name for collection in collections_response.collections]
        return collection_name in existing_names


def extract_existing_vector_size(collection_info: Any) -> Optional[int]:
    try:
        vectors_config = collection_info.config.params.vectors
    except Exception:
        return None

    if hasattr(vectors_config, "size"):
        return vectors_config.size

    if isinstance(vectors_config, dict) and vectors_config:
        first_value = next(iter(vectors_config.values()))
        if hasattr(first_value, "size"):
            return first_value.size
        if isinstance(first_value, dict):
            return first_value.get("size")

    return None


def initialize_or_validate_collection(
    client: QdrantClient,
    collection_name: str,
    vector_size: int,
) -> Dict[str, Any]:
    if not collection_exists(client, collection_name):
        print(f"Creating new Qdrant collection: {collection_name}")
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(
                size=vector_size,
                distance=Distance.COSINE
            ),
        )
        collection_info = client.get_collection(collection_name)
        print("Collection created successfully.")
        return {
            "collection_name": collection_name,
            "created": True,
            "vector_size": extract_existing_vector_size(collection_info),
            "points_count": collection_info.points_count,
            "status": str(collection_info.status),
        }

    print(f"Collection '{collection_name}' already exists.")
    collection_info = client.get_collection(collection_name)
    existing_vector_size = extract_existing_vector_size(collection_info)

    if existing_vector_size is not None and existing_vector_size != vector_size:
        raise ValueError(
            f"Existing collection vector size mismatch. "
            f"Expected {vector_size}, found {existing_vector_size}."
        )

    print("Existing collection is compatible with current embedding configuration.")
    return {
        "collection_name": collection_name,
        "created": False,
        "vector_size": existing_vector_size,
        "points_count": collection_info.points_count,
        "status": str(collection_info.status),
    }


print("Collection helpers ready.")

Collection helpers ready.


## 15. Initialize or Validate the Target Qdrant Collection

In [16]:
collection_init_summary = initialize_or_validate_collection(
    client=qdrant_client,
    collection_name=CONFIG["collection_name"],
    vector_size=CONFIG["embed_dimension"],
)

print("Collection initialization summary:")
print(json.dumps(collection_init_summary, indent=2, ensure_ascii=False))

Creating new Qdrant collection: financial_chunks_xbrl_context_aware_v1
Collection created successfully.
Collection initialization summary:
{
  "collection_name": "financial_chunks_xbrl_context_aware_v1",
  "created": true,
  "vector_size": 1024,
  "points_count": 0,
  "status": "green"
}


## 16. Define Deterministic Point ID Strategy

In [17]:
def generate_deterministic_point_id(chunk_id: str) -> str:
    return hashlib.md5(chunk_id.encode("utf-8")).hexdigest()


print("Deterministic point ID helper ready.")

Deterministic point ID helper ready.


## 17. Define Qdrant Payload and Point Builders

In [18]:
def build_qdrant_payload(chunk: Dict[str, Any]) -> Dict[str, Any]:
    metadata = chunk["metadata"]
    source_file = str(metadata.get("source_file", "unknown_source.json"))

    payload = {
        "chunk_id": chunk["chunk_id"],
        "text": chunk["text"],
        "text_preview": chunk["text"][:500],
        "company_name": sanitize_payload_value(metadata.get("company_name")),
        "currency": sanitize_payload_value(metadata.get("currency")),
        "rounding": sanitize_payload_value(metadata.get("rounding")),
        "report_type": sanitize_payload_value(metadata.get("report_type")),
        "source_file": source_file,
        "source_file_stem": Path(source_file).stem,
        "taxonomy": sanitize_payload_value(metadata.get("taxonomy")),
        "section_key": sanitize_payload_value(metadata.get("section_key")),
        "section_name": sanitize_payload_value(metadata.get("section_name")),
        "chunk_type": sanitize_payload_value(metadata.get("chunk_type")),
        "reporting_period": sanitize_payload_value(metadata.get("reporting_period")),
        "context_type": sanitize_payload_value(metadata.get("context_type")),
        "context_ids": sanitize_payload_value(metadata.get("context_ids", [])),
        "start_date": sanitize_payload_value(metadata.get("start_date")),
        "end_date": sanitize_payload_value(metadata.get("end_date")),
        "instant_date": sanitize_payload_value(metadata.get("instant_date")),
        "dimension_members": sanitize_payload_value(metadata.get("dimension_members", [])),
        "source_tags": sanitize_payload_value(metadata.get("source_tags", [])),
        "unified_fields": sanitize_payload_value(metadata.get("unified_fields", [])),
        "context_signature": sanitize_payload_value(metadata.get("context_signature")),
        "chunk_index": sanitize_payload_value(metadata.get("chunk_index")),
        "word_count": sanitize_payload_value(metadata.get("word_count")),
        "markdown_style": sanitize_payload_value(metadata.get("markdown_style")),
        "is_narrative_chunk": metadata.get("chunk_type") == "narrative_disclosure",
        "is_unified_chunk": metadata.get("chunk_type") == "unified_metric_group",
        "is_metadata_chunk": metadata.get("chunk_type") == "filing_metadata",
        "has_dimensions": bool(metadata.get("dimension_members")),
    }

    return payload


def build_qdrant_point(chunk: Dict[str, Any], vector: List[float]) -> PointStruct:
    payload = build_qdrant_payload(chunk)
    return PointStruct(
        id=generate_deterministic_point_id(chunk["chunk_id"]),
        vector=vector,
        payload=payload,
    )


print("Qdrant payload and point builders ready.")

Qdrant payload and point builders ready.


## 18. Define the Bedrock Embedding Function with Retry Logic

In [19]:
def get_embedding(prepared_text: str, max_retries: int = 5) -> Tuple[List[float], Dict[str, Any]]:
    payload = {
        "inputText": prepared_text,
        "dimensions": CONFIG["embed_dimension"],
        "normalize": CONFIG["embed_normalize"],
    }

    body = json.dumps(payload)
    throttled_retries = 0
    request_start_time = time.time()

    for attempt in range(max_retries):
        try:
            response = bedrock_client.invoke_model(
                body=body,
                modelId=CONFIG["embed_model_id"],
                accept="application/json",
                contentType="application/json"
            )

            response_body = json.loads(response["body"].read())
            embedding = response_body.get("embedding")

            if not embedding:
                raise ValueError("Embedding response is missing the 'embedding' field.")

            if len(embedding) != CONFIG["embed_dimension"]:
                raise ValueError(
                    f"Embedding dimension mismatch. "
                    f"Expected {CONFIG['embed_dimension']}, got {len(embedding)}."
                )

            return embedding, {
                "attempts_used": attempt + 1,
                "throttled_retries": throttled_retries,
                "latency_seconds": round(time.time() - request_start_time, 4),
            }

        except ClientError as e:
            error_code = e.response.get("Error", {}).get("Code", "")

            if error_code in {"ThrottlingException", "TooManyRequestsException"}:
                throttled_retries += 1
                sleep_time = 2 ** attempt
                print(
                    f"Throttling detected. Retrying in {sleep_time} seconds "
                    f"(attempt {attempt + 1}/{max_retries})..."
                )
                time.sleep(sleep_time)
            else:
                raise RuntimeError(f"Bedrock embedding request failed: {e}")

        except Exception as e:
            raise RuntimeError(f"Unexpected embedding error: {e}")

    raise TimeoutError(f"Failed to get embedding after {max_retries} retries.")

## 19. Run a Small Embedding Smoke Test

In [20]:
if not ingestion_chunks:
    raise ValueError("No ingestion-ready chunks available for embedding smoke test.")

sample_chunk = ingestion_chunks[0]
sample_prepared_text, sample_text_prep_meta = prepare_text_for_embedding(
    sample_chunk["text"],
    max_chars=CONFIG["embed_text_max_chars"],
)

print("Running embedding smoke test...\n")
sample_vector, sample_embedding_meta = get_embedding(
    sample_prepared_text,
    max_retries=CONFIG["embed_max_retries"],
)

print("Embedding smoke test successful.")
print("Sample chunk ID:", sample_chunk["chunk_id"])
print("Prepared text chars:", sample_text_prep_meta["prepared_char_count"])
print("Prepared text words:", sample_text_prep_meta["prepared_word_count"])
print("Was truncated:", sample_text_prep_meta["was_truncated"])
print("Vector length:", len(sample_vector))
print("Expected length:", CONFIG["embed_dimension"])
print("Attempts used:", sample_embedding_meta["attempts_used"])
print("Latency (seconds):", sample_embedding_meta["latency_seconds"])
print("Sample vector values:", sample_vector[:5])

Running embedding smoke test...

Embedding smoke test successful.
Sample chunk ID: gi-117338-1350247-17012025074725-f7b38fa1cd4c
Prepared text chars: 405
Prepared text words: 46
Was truncated: False
Vector length: 1024
Expected length: 1024
Attempts used: 1
Latency (seconds): 6.0313
Sample vector values: [-0.06709370762109756, 0.029960831627249718, 0.037363309413194656, -0.01330247987061739, 0.02453411929309368]


## 20. Define Qdrant Batch Upsert Helper

In [21]:
def build_failure_record(chunk: Dict[str, Any], error_category: str, error_message: str) -> Dict[str, Any]:
    metadata = chunk.get("metadata", {})
    return {
        "chunk_id": chunk.get("chunk_id"),
        "source_file": metadata.get("source_file"),
        "company_name": metadata.get("company_name"),
        "taxonomy": metadata.get("taxonomy"),
        "chunk_type": metadata.get("chunk_type"),
        "section_key": metadata.get("section_key"),
        "reporting_period": metadata.get("reporting_period"),
        "error_category": error_category,
        "error": str(error_message),
    }


def upsert_points_batch(
    client: QdrantClient,
    collection_name: str,
    batch_items: List[Dict[str, Any]],
) -> Dict[str, Any]:
    if not batch_items:
        return {
            "upserted_count": 0,
            "failed_records": [],
            "used_single_point_fallback": False,
        }

    points = [item["point"] for item in batch_items]

    try:
        client.upsert(
            collection_name=collection_name,
            points=points,
        )
        return {
            "upserted_count": len(points),
            "failed_records": [],
            "used_single_point_fallback": False,
        }

    except Exception as batch_error:
        failed_records = []
        successful_count = 0

        for item in batch_items:
            try:
                client.upsert(
                    collection_name=collection_name,
                    points=[item["point"]],
                )
                successful_count += 1

            except Exception as single_error:
                failed_record = build_failure_record(
                    chunk=item["chunk"],
                    error_category="upsert_error",
                    error_message=f"Batch error: {batch_error} | Single upsert error: {single_error}",
                )
                failed_records.append(failed_record)

        return {
            "upserted_count": successful_count,
            "failed_records": failed_records,
            "used_single_point_fallback": True,
        }


print("Batch upsert helper ready.")

Batch upsert helper ready.


## 21. Generate Embeddings and Upsert Points in Batches

In [22]:
points_batch: List[Dict[str, Any]] = []
embedding_failures: List[Dict[str, Any]] = []
payload_failures: List[Dict[str, Any]] = []
upsert_failures: List[Dict[str, Any]] = []

processed_count = 0
embedded_success_count = 0
payload_success_count = 0
total_upserted = 0

text_truncation_count = 0
embedding_attempts_total = 0
embedding_throttle_retries_total = 0
embedding_latency_total = 0.0
batch_upsert_calls = 0
fallback_batch_count = 0

ingestion_start_time = time.time()

print(f"Starting ingestion for {len(ingestion_chunks)} chunks...\n")

for chunk in tqdm(ingestion_chunks, desc="Embedding and Upserting Chunks"):
    processed_count += 1

    prepared_text, text_prep_meta = prepare_text_for_embedding(
        chunk["text"],
        max_chars=CONFIG["embed_text_max_chars"],
    )

    if text_prep_meta["was_truncated"]:
        text_truncation_count += 1

    try:
        vector, embedding_meta = get_embedding(
            prepared_text,
            max_retries=CONFIG["embed_max_retries"],
        )
        embedded_success_count += 1
        embedding_attempts_total += embedding_meta["attempts_used"]
        embedding_throttle_retries_total += embedding_meta["throttled_retries"]
        embedding_latency_total += embedding_meta["latency_seconds"]

    except Exception as e:
        embedding_failures.append(
            build_failure_record(
                chunk=chunk,
                error_category="embedding_error",
                error_message=e,
            )
        )
        continue

    try:
        point = build_qdrant_point(chunk, vector)
        points_batch.append({
            "point": point,
            "chunk": chunk,
        })
        payload_success_count += 1

    except Exception as e:
        payload_failures.append(
            build_failure_record(
                chunk=chunk,
                error_category="payload_error",
                error_message=e,
            )
        )
        continue

    if len(points_batch) >= CONFIG["batch_size"]:
        batch_result = upsert_points_batch(
            client=qdrant_client,
            collection_name=CONFIG["collection_name"],
            batch_items=points_batch,
        )
        batch_upsert_calls += 1
        if batch_result["used_single_point_fallback"]:
            fallback_batch_count += 1

        total_upserted += batch_result["upserted_count"]
        upsert_failures.extend(batch_result["failed_records"])
        points_batch = []

if points_batch:
    batch_result = upsert_points_batch(
        client=qdrant_client,
        collection_name=CONFIG["collection_name"],
        batch_items=points_batch,
    )
    batch_upsert_calls += 1
    if batch_result["used_single_point_fallback"]:
        fallback_batch_count += 1

    total_upserted += batch_result["upserted_count"]
    upsert_failures.extend(batch_result["failed_records"])
    points_batch = []

ingestion_elapsed_seconds = round(time.time() - ingestion_start_time, 2)

print("\nIngestion completed.")
print("Processed chunks         :", processed_count)
print("Embedded successfully    :", embedded_success_count)
print("Payloads built successfully:", payload_success_count)
print("Total points upserted    :", total_upserted)
print("Embedding failures       :", len(embedding_failures))
print("Payload failures         :", len(payload_failures))
print("Upsert failures          :", len(upsert_failures))
print("Text truncations         :", text_truncation_count)
print("Elapsed time (seconds)   :", ingestion_elapsed_seconds)

Starting ingestion for 1383 chunks...



Embedding and Upserting Chunks: 100%|██████████| 1383/1383 [19:02<00:00,  1.21it/s]


Ingestion completed.
Processed chunks         : 1383
Embedded successfully    : 1383
Payloads built successfully: 1383
Total points upserted    : 1383
Embedding failures       : 0
Payload failures         : 0
Upsert failures          : 0
Text truncations         : 0
Elapsed time (seconds)   : 1142.37


## 22. Review Failed Chunk Records

In [23]:
all_failed_chunks = embedding_failures + payload_failures + upsert_failures
failed_chunk_category_counter = Counter(item["error_category"] for item in all_failed_chunks)

print("Failure review summary")
print("-" * 80)
print("Total failed chunks:", len(all_failed_chunks))

if failed_chunk_category_counter:
    print("\nFailures by category:")
    for category, count in failed_chunk_category_counter.most_common():
        print(f"- {category}: {count}")

if all_failed_chunks:
    print("\nSample failed chunk records:")
    for failed_chunk in all_failed_chunks[:10]:
        print(json.dumps(failed_chunk, indent=2, ensure_ascii=False))
else:
    print("No failed chunks found.")

Failure review summary
--------------------------------------------------------------------------------
Total failed chunks: 0
No failed chunks found.


## 23. Verify Qdrant Collection State After Ingestion

In [24]:
collection_info = qdrant_client.get_collection(CONFIG["collection_name"])

collection_summary = {
    "collection_name": CONFIG["collection_name"],
    "status": str(collection_info.status),
    "points_count": collection_info.points_count,
    "indexed_vectors_count": collection_info.indexed_vectors_count,
    "vector_size": extract_existing_vector_size(collection_info),
}

print("Qdrant collection verification")
print("-" * 80)
print(json.dumps(collection_summary, indent=2, ensure_ascii=False))

Qdrant collection verification
--------------------------------------------------------------------------------
{
  "collection_name": "financial_chunks_xbrl_context_aware_v1",
  "status": "green",
  "points_count": 1383,
  "indexed_vectors_count": 0,
  "vector_size": 1024
}


## 24. Build Final Ingestion Audit Payload

In [25]:
average_embedding_latency = round(
    embedding_latency_total / embedded_success_count, 4
) if embedded_success_count else None

average_embedding_attempts = round(
    embedding_attempts_total / embedded_success_count, 4
) if embedded_success_count else None

final_ingestion_audit = {
    "pipeline_version": "qdrant_ingestion_v2",
    "config": {
        key: value for key, value in CONFIG.items()
    },
    "input_summary": {
        "input_file_path": str(input_file_path.resolve()),
        "input_audit_file_path": str(input_audit_file_path.resolve()),
        "loaded_chunk_records": len(structured_chunks),
        "valid_chunks": len(valid_chunks),
        "schema_issues": len(schema_issues),
        "duplicate_chunk_ids": len(duplicate_chunk_ids),
        "ingestion_chunks": len(ingestion_chunks),
    },
    "upstream_chunk_audit_summary": chunk_audit_summary,
    "filter_summary": filter_summary,
    "distribution": ingestion_distribution,
    "embedding_metrics": {
        "embedded_success_count": embedded_success_count,
        "embedding_attempts_total": embedding_attempts_total,
        "embedding_throttle_retries_total": embedding_throttle_retries_total,
        "average_embedding_attempts": average_embedding_attempts,
        "average_embedding_latency_seconds": average_embedding_latency,
        "text_truncation_count": text_truncation_count,
    },
    "upsert_metrics": {
        "payload_success_count": payload_success_count,
        "total_upserted": total_upserted,
        "batch_upsert_calls": batch_upsert_calls,
        "fallback_batch_count": fallback_batch_count,
    },
    "failure_metrics": {
        "embedding_failures": len(embedding_failures),
        "payload_failures": len(payload_failures),
        "upsert_failures": len(upsert_failures),
        "total_failed_chunks": len(all_failed_chunks),
        "by_category": dict(failed_chunk_category_counter),
    },
    "collection_summary": collection_summary,
    "runtime": {
        "ingestion_elapsed_seconds": ingestion_elapsed_seconds,
    },
}

print("Final ingestion audit payload prepared.")
print(json.dumps({
    "loaded_chunk_records": final_ingestion_audit["input_summary"]["loaded_chunk_records"],
    "ingestion_chunks": final_ingestion_audit["input_summary"]["ingestion_chunks"],
    "total_upserted": final_ingestion_audit["upsert_metrics"]["total_upserted"],
    "total_failed_chunks": final_ingestion_audit["failure_metrics"]["total_failed_chunks"],
}, indent=2, ensure_ascii=False))

Final ingestion audit payload prepared.
{
  "loaded_chunk_records": 1383,
  "ingestion_chunks": 1383,
  "total_upserted": 1383,
  "total_failed_chunks": 0
}


## 25. Save Ingestion Audit Outputs

In [26]:
ingestion_audit_output_path = save_json(
    final_ingestion_audit,
    Path(CONFIG["ingestion_audit_output_path"]),
)

failed_chunks_output_path = save_json(
    all_failed_chunks,
    Path(CONFIG["failed_chunks_output_path"]),
)

schema_issues_output_path = save_json(
    schema_issues,
    Path(CONFIG["schema_issues_output_path"]),
)

print("Ingestion audit outputs saved successfully.")
print("Audit output path       :", ingestion_audit_output_path.resolve())
print("Failed chunks output    :", failed_chunks_output_path.resolve())
print("Schema issues output    :", schema_issues_output_path.resolve())

Ingestion audit outputs saved successfully.
Audit output path       : C:\Users\Home\llmops-nse-rag\data\processed\audits\qdrant_ingestion_audit_v1.json
Failed chunks output    : C:\Users\Home\llmops-nse-rag\data\processed\audits\qdrant_failed_chunks_v1.json
Schema issues output    : C:\Users\Home\llmops-nse-rag\data\processed\audits\qdrant_schema_issues_v1.json


## 26. Print Final Ingestion Summary

In [27]:
print("Final ingestion summary")
print("-" * 80)
print("Input file                     :", input_file_path.resolve())
print("Target collection              :", CONFIG["collection_name"])
print("Total chunk records loaded     :", len(structured_chunks))
print("Valid chunks after schema check:", len(valid_chunks))
print("Final ingestion chunks         :", len(ingestion_chunks))
print("Embedded successfully          :", embedded_success_count)
print("Payloads built successfully    :", payload_success_count)
print("Total points upserted          :", total_upserted)
print("Total failed chunks            :", len(all_failed_chunks))
print("Schema issues                  :", len(schema_issues))
print("Duplicate chunk IDs            :", len(duplicate_chunk_ids))
print("Elapsed time (seconds)         :", ingestion_elapsed_seconds)
print("Audit output file              :", ingestion_audit_output_path.name)
print("Failed chunks output file      :", failed_chunks_output_path.name)
print("Schema issues output file      :", schema_issues_output_path.name)

Final ingestion summary
--------------------------------------------------------------------------------
Input file                     : C:\Users\Home\llmops-nse-rag\data\processed\chunks\financial_structured_chunks_v2.json
Target collection              : financial_chunks_xbrl_context_aware_v1
Total chunk records loaded     : 1383
Valid chunks after schema check: 1383
Final ingestion chunks         : 1383
Embedded successfully          : 1383
Payloads built successfully    : 1383
Total points upserted          : 1383
Total failed chunks            : 0
Schema issues                  : 0
Duplicate chunk IDs            : 0
Elapsed time (seconds)         : 1142.37
Audit output file              : qdrant_ingestion_audit_v1.json
Failed chunks output file      : qdrant_failed_chunks_v1.json
Schema issues output file      : qdrant_schema_issues_v1.json
